In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import torch
import torch.nn as nn

from torchvision import models

from data.dataloader import get_dataloaders

In [3]:
train_loader, val_loader, test_loader, class_names = get_dataloaders()
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

3.7%

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\TANISHA/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
in_features = model.classifier[1].in_features
num_classes = len(class_names)
model.classifier[1] = torch.nn.Linear(in_features, num_classes)

# Move model to device
model = model.to(device)
print("Modified Classifier:", model.classifier)

# 5. Loss and Optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

cpu
Modified Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=30, bias=True)
)


In [5]:
# 6. Training Configuration
epochs = 10
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
best_val_acc = 0

# 7. Training Loop
for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

In [6]:
# ---------------- VALIDATION ----------------
model.eval()
running_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_loss = running_loss / len(val_loader)
val_acc = 100 * correct / total
val_losses.append(val_loss)
val_accuracies.append(val_acc)

print(
    f"Epoch {epoch+1}/{epochs} | "
    f"Train Loss: {train_loss:.4f} | "
    f"Train Acc: {train_acc:.2f}% | "
    f"Val Loss: {val_loss:.4f} | "
    f"Val Acc: {val_acc:.2f}%"
)

# Save the best model checkpoint
if val_acc > best_val_acc:
    best_val_acc = val_acc
    os.makedirs("../models", exist_ok=True)
    torch.save(model.state_dict(), "../models/efficientnet_b0_best.pth")

Epoch 10/10 | Train Loss: 0.1087 | Train Acc: 96.09% | Val Loss: 0.5070 | Val Acc: 85.78%
